<a href="https://colab.research.google.com/github/adonezukanma/Data-Software-Engineer/blob/main/Netflix_Clone_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
# Step 1: Install all required packages
print("📦 Installing dependencies...")

!pip install fastapi uvicorn python-dotenv requests python-multipart nest-asyncio pyngrok > /dev/null 2>&1
!apt-get update > /dev/null 2>&1
!apt-get install -y nodejs npm > /dev/null 2>&1

print("✅ Dependencies installed successfully!")

📦 Installing dependencies...
✅ Dependencies installed successfully!


In [81]:
#Step 2: Setup Ngrok for public URLs
print("🌐 Setting up Ngrok...")

from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

# Get your free token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "34AjV5YWfyZhMrdhmMDgezHGLi5_7fKxxZh6uZ5cWCSCNt9RX"  # Replace with your actual token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("✅ Ngrok configured!")

🌐 Setting up Ngrok...
✅ Ngrok configured!


In [82]:
# Step 3: Create folder structure
print("📁 Creating project structure...")

import os

# Create directories
os.makedirs("netflix-clone/backend/app/routes", exist_ok=True)
os.makedirs("netflix-clone/backend/app/services", exist_ok=True)
os.makedirs("netflix-clone/frontend/src/components", exist_ok=True)
os.makedirs("netflix-clone/frontend/src/hooks", exist_ok=True)

# Create __init__.py files
open("netflix-clone/backend/app/__init__.py", "w").close()
open("netflix-clone/backend/app/routes/__init__.py", "w").close()
open("netflix-clone/backend/app/services/__init__.py", "w").close()

print("✅ Project structure created!")

📁 Creating project structure...
✅ Project structure created!


In [83]:
# Step 4: Create backend configuration
print("🔧 Creating backend files...")

# backend/app/config.py
config_content = '''import os

class Settings:
    TMDB_API_KEYS = os.getenv("TMDB_API_KEYS", "").split(",")
    TMDB_BASE_URL = "https://api.themoviedb.org/3"
    TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p"

    def __init__(self):
        self.current_key_index = 0

    def get_tmdb_api_key(self):
        if not self.TMDB_API_KEYS:
            raise ValueError("No TMDB API keys configured")

        key = self.TMDB_API_KEYS[self.current_key_index]
        self.current_key_index = (self.current_key_index + 1) % len(self.TMDB_API_KEYS)
        return key

settings = Settings()
'''

with open("netflix-clone/backend/app/config.py", "w") as f:
    f.write(config_content)

# backend/app/services/tmdb.py
tmdb_content = '''import requests
from ..config import settings

class TMDBService:
    def __init__(self):
        self.base_url = settings.TMDB_BASE_URL
        self.image_base_url = settings.TMDB_IMAGE_BASE_URL

    def _make_request(self, endpoint, params=None):
        url = f"{self.base_url}/{endpoint}"
        api_key = settings.get_tmdb_api_key()

        default_params = {
            "api_key": api_key,
            "language": "en-US"
        }
        if params:
            default_params.update(params)

        try:
            response = requests.get(url, params=default_params)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"TMDB API Error: {e}")
            return None

    def get_trending_movies(self):
        return self._make_request("trending/movie/week")

    def get_popular_movies(self):
        return self._make_request("movie/popular")

    def get_top_rated_movies(self):
        return self._make_request("movie/top_rated")

    def get_now_playing_movies(self):
        return self._make_request("movie/now_playing")

    def get_upcoming_movies(self):
        return self._make_request("movie/upcoming")

    def get_movie_details(self, movie_id):
        return self._make_request(f"movie/{movie_id}")

    def get_movie_trailer(self, movie_id):
        data = self._make_request(f"movie/{movie_id}/videos")
        if data and "results" in data:
            for video in data["results"]:
                if video["type"] == "Trailer" and video["site"] == "YouTube":
                    return video
        return None

    def get_movie_backdrop(self, movie, size="w1280"):
        if movie.get("backdrop_path"):
            return f"{self.image_base_url}/{size}{movie['backdrop_path']}"
        return None

    def get_movie_poster(self, movie, size="w500"):
        if movie.get("poster_path"):
            return f"{self.image_base_url}/{size}{movie['poster_path']}"
        return None

tmdb_service = TMDBService()
'''

with open("netflix-clone/backend/app/services/tmdb.py", "w") as f:
    f.write(tmdb_content)

# backend/app/routes/content.py
routes_content = '''from fastapi import APIRouter, HTTPException
from ..services.tmdb import tmdb_service

router = APIRouter()

@router.get("/featured")
async def get_featured_content():
    trending = tmdb_service.get_trending_movies()
    if trending and "results" in trending and trending["results"]:
        featured_movie = trending["results"][0]
        featured_movie["backdrop_url"] = tmdb_service.get_movie_backdrop(featured_movie)
        featured_movie["poster_url"] = tmdb_service.get_movie_poster(featured_movie)
        return featured_movie
    raise HTTPException(status_code=404, detail="No featured content found")

@router.get("/rows")
async def get_content_rows():
    categories = {
        "Trending Now": tmdb_service.get_trending_movies(),
        "Popular Movies": tmdb_service.get_popular_movies(),
        "Top Rated": tmdb_service.get_top_rated_movies(),
        "Now Playing": tmdb_service.get_now_playing_movies(),
        "Upcoming": tmdb_service.get_upcoming_movies()
    }

    rows = []
    for category_name, data in categories.items():
        if data and "results" in data:
            row_data = {
                "title": category_name,
                "movies": []
            }

            for movie in data["results"][:20]:
                movie_data = {
                    "id": movie["id"],
                    "title": movie.get("title", "Unknown"),
                    "overview": movie.get("overview", ""),
                    "poster_url": tmdb_service.get_movie_poster(movie),
                    "backdrop_url": tmdb_service.get_movie_backdrop(movie),
                    "vote_average": movie.get("vote_average", 0),
                    "release_date": movie.get("release_date", "")
                }
                row_data["movies"].append(movie_data)

            rows.append(row_data)

    return rows

@router.get("/trailer/{movie_id}")
async def get_movie_trailer(movie_id: int):
    trailer = tmdb_service.get_movie_trailer(movie_id)
    if trailer:
        return {
            "youtube_key": trailer["key"],
            "name": trailer["name"],
            "type": trailer["type"]
        }
    raise HTTPException(status_code=404, detail="Trailer not found")
'''

with open("netflix-clone/backend/app/routes/content.py", "w") as f:
    f.write(routes_content)

# backend/app/main.py
main_content = '''from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from .routes.content import router as content_router

app = FastAPI(title="Netflix Clone API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(content_router, prefix="/api/content", tags=["content"])

@app.get("/")
async def root():
    return {"message": "Netflix Clone API"}

@app.get("/health")
async def health_check():
    return {"status": "healthy"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("netflix-clone/backend/app/main.py", "w") as f:
    f.write(main_content)

# backend/requirements.txt
requirements_content = '''fastapi==0.104.1
uvicorn==0.24.0
python-dotenv==1.0.0
requests==2.31.0
python-multipart==0.0.6
'''

with open("netflix-clone/backend/requirements.txt", "w") as f:
    f.write(requirements_content)

print("✅ Backend files created!")

🔧 Creating backend files...
✅ Backend files created!


In [84]:
# Step 5: Create frontend files
print("🎨 Creating frontend files...")

# frontend/package.json
package_json = '''{
  "name": "netflix-clone-frontend",
  "private": true,
  "version": "1.0.0",
  "type": "module",
  "scripts": {
    "dev": "vite",
    "build": "vite build",
    "preview": "vite preview",
    "serve": "vite preview --host 0.0.0.0 --port 3000"
  },
  "dependencies": {
    "react": "^18.2.0",
    "react-dom": "^18.2.0"
  },
  "devDependencies": {
    "vite": "^4.4.5",
    "@vitejs/plugin-react": "^4.0.3",
    "autoprefixer": "^10.4.14",
    "postcss": "^8.4.27",
    "tailwindcss": "^3.3.0"
  }
}
'''

with open("netflix-clone/frontend/package.json", "w") as f:
    f.write(package_json)

# frontend/index.html
html_content = '''<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <link rel="icon" type="image/svg+xml" href="/vite.svg" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>Netflix Clone</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.jsx"></script>
  </body>
</html>
'''

with open("netflix-clone/frontend/index.html", "w") as f:
    f.write(html_content)

# frontend/vite.config.js
vite_config = '''import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

export default defineConfig({
  plugins: [react()],
  server: {
    host: true,
    port: 3000
  },
  define: {
    'process.env': {}
  }
})
'''

with open("netflix-clone/frontend/vite.config.js", "w") as f:
    f.write(vite_config)

# frontend/tailwind.config.js
tailwind_config = '''/** @type {import('tailwindcss').Config} */
export default {
  content: [
    "./index.html",
    "./src/**/*.{js,ts,jsx,tsx}",
  ],
  theme: {
    extend: {
      colors: {
        netflix: {
          black: '#141414',
          red: '#E50914',
          gray: '#2F2F2F',
          lightgray: '#808080'
        }
      }
    },
  },
  plugins: [],
}
'''

with open("netflix-clone/frontend/tailwind.config.js", "w") as f:
    f.write(tailwind_config)

# frontend/postcss.config.js
postcss_config = '''export default {
  plugins: {
    tailwindcss: {},
    autoprefixer: {},
  },
}
'''

with open("netflix-clone/frontend/postcss.config.js", "w") as f:
    f.write(postcss_config)

# frontend/src/main.jsx
main_jsx = '''import React from 'react'
import ReactDOM from 'react-dom/client'
import App from './App.jsx'
import './index.css'

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <App />
  </React.StrictMode>,
)
'''

with open("netflix-clone/frontend/src/main.jsx", "w") as f:
    f.write(main_jsx)

# frontend/src/index.css
index_css = '''@tailwind base;
@tailwind components;
@tailwind utilities;

.scrollbar-hide {
  -ms-overflow-style: none;
  scrollbar-width: none;
}
.scrollbar-hide::-webkit-scrollbar {
  display: none;
}

.line-clamp-3 {
  display: -webkit-box;
  -webkit-line-clamp: 3;
  -webkit-box-orient: vertical;
  overflow: hidden;
}

body {
  margin: 0;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Roboto', 'Oxygen',
    'Ubuntu', 'Cantarell', 'Fira Sans', 'Droid Sans', 'Helvetica Neue',
    sans-serif;
  -webkit-font-smoothing: antialiased;
  -moz-osx-font-smoothing: grayscale;
  background-color: #141414;
  color: white;
}
'''

with open("netflix-clone/frontend/src/index.css", "w") as f:
    f.write(index_css)

print("✅ Frontend base files created!")

🎨 Creating frontend files...
✅ Frontend base files created!


In [85]:
# Step 6: Create React components
print("⚛️ Creating React components...")

# frontend/src/hooks/useFetch.js
usefetch_content = '''import { useState, useEffect } from 'react';

const useFetch = (endpoint) => {
  const [data, setData] = useState(null);
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState(null);

  useEffect(() => {
    const fetchData = async () => {
      if (!endpoint) return;

      try {
        setLoading(true);
        const response = await fetch(endpoint);

        if (!response.ok) {
          throw new Error(`HTTP error! status: ${response.status}`);
        }

        const result = await response.json();
        setData(result);
        setError(null);
      } catch (err) {
        setError(err.message);
        console.error('Fetch error:', err);
      } finally {
        setLoading(false);
      }
    };

    fetchData();
  }, [endpoint]);

  return { data, loading, error };
};

export default useFetch;
'''

with open("netflix-clone/frontend/src/hooks/useFetch.js", "w") as f:
    f.write(usefetch_content)

# frontend/src/components/Navbar.jsx
navbar_content = '''import React, { useState, useEffect } from 'react';

const Navbar = () => {
  const [isScrolled, setIsScrolled] = useState(false);

  useEffect(() => {
    const handleScroll = () => {
      setIsScrolled(window.scrollY > 0);
    };

    window.addEventListener('scroll', handleScroll);
    return () => window.removeEventListener('scroll', handleScroll);
  }, []);

  return (
    <nav className={`fixed top-0 w-full z-50 transition-all duration-500 ${
      isScrolled ? 'bg-netflix-black' : 'bg-transparent'
    }`}>
      <div className="flex items-center justify-between px-8 py-4">
        <div className="flex items-center space-x-8">
          <div className="text-netflix-red font-bold text-4xl">NETFLIX</div>
          <div className="hidden md:flex space-x-6">
            <a href="#" className="text-white hover:text-gray-300 transition">Home</a>
            <a href="#" className="text-white hover:text-gray-300 transition">TV Shows</a>
            <a href="#" className="text-white hover:text-gray-300 transition">Movies</a>
            <a href="#" className="text-white hover:text-gray-300 transition">New & Popular</a>
            <a href="#" className="text-white hover:text-gray-300 transition">My List</a>
          </div>
        </div>

        <div className="flex items-center space-x-4">
          <button className="text-white hover:text-gray-300 transition">
            <svg className="w-6 h-6" fill="none" stroke="currentColor" viewBox="0 0 24 24">
              <path strokeLinecap="round" strokeLinejoin="round" strokeWidth={2} d="M21 21l-6-6m2-5a7 7 0 11-14 0 7 7 0 0114 0z" />
            </svg>
          </button>
          <div className="text-white">Kids</div>
          <div className="w-8 h-8 bg-netflix-gray rounded flex items-center justify-center">
            <span className="text-white text-sm">👤</span>
          </div>
        </div>
      </div>
    </nav>
  );
};

export default Navbar;
'''

with open("netflix-clone/frontend/src/components/Navbar.jsx", "w") as f:
    f.write(navbar_content)

# frontend/src/components/Hero.jsx
hero_content = '''import React from 'react';
import useFetch from '../hooks/useFetch';

const Hero = ({ onPlayTrailer }) => {
  const { data: featuredMovie, loading, error } = useFetch('/api/content/featured');

  if (loading) {
    return (
      <div className="h-screen bg-netflix-black flex items-center justify-center">
        <div className="text-white text-xl">Loading...</div>
      </div>
    );
  }

  if (error || !featuredMovie) {
    return (
      <div className="h-screen bg-netflix-black flex items-center justify-center">
        <div className="text-white text-xl">Error loading featured content</div>
      </div>
    );
  }

  const backgroundStyle = featuredMovie.backdrop_url
    ? {
        backgroundImage: `linear-gradient(to top, rgba(0, 0, 0, 0.8) 0%, rgba(0, 0, 0, 0.4) 60%, rgba(0, 0, 0, 0.8) 100%), url(${featuredMovie.backdrop_url})`,
        backgroundSize: 'cover',
        backgroundPosition: 'center'
      }
    : { backgroundColor: '#141414' };

  return (
    <div
      className="h-screen bg-cover bg-center flex items-end pb-16"
      style={backgroundStyle}
    >
      <div className="container mx-auto px-8">
        <div className="max-w-2xl">
          <h1 className="text-5xl md:text-6xl font-bold text-white mb-4">
            {featuredMovie.title}
          </h1>
          <p className="text-lg text-white mb-6 line-clamp-3">
            {featuredMovie.overview}
          </p>
          <div className="flex space-x-4">
            <button
              onClick={() => onPlayTrailer(featuredMovie.id)}
              className="bg-white text-black px-6 py-2 rounded flex items-center space-x-2 hover:bg-opacity-80 transition"
            >
              <span>▶</span>
              <span>Play</span>
            </button>
            <button className="bg-netflix-gray bg-opacity-70 text-white px-6 py-2 rounded flex items-center space-x-2 hover:bg-opacity-50 transition">
              <span>ℹ</span>
              <span>More Info</span>
            </button>
          </div>
        </div>
      </div>
    </div>
  );
};

export default Hero;
'''

with open("netflix-clone/frontend/src/components/Hero.jsx", "w") as f:
    f.write(hero_content)

# frontend/src/components/Row.jsx
row_content = '''import React, { useRef, useState } from 'react';

const Row = ({ title, movies }) => {
  const rowRef = useRef(null);
  const [isMoved, setIsMoved] = useState(false);

  const handleClick = (direction) => {
    setIsMoved(true);

    if (rowRef.current) {
      const { scrollLeft, clientWidth } = rowRef.current;

      const scrollTo = direction === 'left'
        ? scrollLeft - clientWidth
        : scrollLeft + clientWidth;

      rowRef.current.scrollTo({ left: scrollTo, behavior: 'smooth' });
    }
  };

  return (
    <div className="relative group">
      <h2 className="text-white text-2xl font-bold mb-4 px-8">{title}</h2>

      <button
        onClick={() => handleClick('left')}
        className={`absolute left-2 top-1/2 z-40 h-12 w-12 bg-netflix-black bg-opacity-70 rounded-full flex items-center justify-center transition opacity-0 group-hover:opacity-100 ${
          !isMoved && 'hidden'
        }`}
      >
        <span className="text-white text-2xl">‹</span>
      </button>

      <button
        onClick={() => handleClick('right')}
        className="absolute right-2 top-1/2 z-40 h-12 w-12 bg-netflix-black bg-opacity-70 rounded-full flex items-center justify-center transition opacity-0 group-hover:opacity-100"
      >
        <span className="text-white text-2xl">›</span>
      </button>

      <div
        ref={rowRef}
        className="flex space-x-4 overflow-x-scroll scrollbar-hide px-8"
      >
        {movies.map((movie) => (
          <div
            key={movie.id}
            className="relative min-w-[200px] md:min-w-[300px] transition-transform duration-300 hover:scale-110 hover:z-30"
          >
            <img
              src={movie.poster_url || 'https://via.placeholder.com/300x450/333333/ffffff?text=No+Image'}
              alt={movie.title}
              className="w-full h-auto rounded object-cover"
              onError={(e) => {
                e.target.src = 'https://via.placeholder.com/300x450/333333/ffffff?text=No+Image';
              }}
            />
            <div className="absolute inset-0 bg-black bg-opacity-0 hover:bg-opacity-70 transition-opacity duration-300 flex items-center justify-center">
              <div className="text-white text-center opacity-0 hover:opacity-100 transition-opacity duration-300 p-4">
                <h3 className="font-bold mb-2">{movie.title}</h3>
                <p className="text-sm line-clamp-3">{movie.overview}</p>
              </div>
            </div>
          </div>
        ))}
      </div>
    </div>
  );
};

export default Row;
'''

with open("netflix-clone/frontend/src/components/Row.jsx", "w") as f:
    f.write(row_content)

# frontend/src/components/TrailerModal.jsx
modal_content = '''import React from 'react';
import useFetch from '../hooks/useFetch';

const TrailerModal = ({ movieId, isOpen, onClose }) => {
  const { data: trailer, loading, error } = useFetch(
    isOpen && movieId ? `/api/content/trailer/${movieId}` : null
  );

  if (!isOpen) return null;

  return (
    <div className="fixed inset-0 z-50 flex items-center justify-center bg-black bg-opacity-80">
      <div className="relative w-full max-w-4xl mx-4">
        <button
          onClick={onClose}
          className="absolute -top-12 right-0 text-white text-2xl hover:text-gray-300 transition"
        >
          ✕
        </button>

        <div className="bg-black rounded-lg overflow-hidden">
          {loading && (
            <div className="h-96 flex items-center justify-center">
              <div className="text-white text-xl">Loading trailer...</div>
            </div>
          )}

          {error && (
            <div className="h-96 flex items-center justify-center">
              <div className="text-white text-xl">Trailer not available</div>
            </div>
          )}

          {trailer && (
            <div className="relative pt-[56.25%]">
              <iframe
                src={`https://www.youtube.com/embed/${trailer.youtube_key}?autoplay=1`}
                className="absolute top-0 left-0 w-full h-full"
                allow="autoplay; encrypted-media"
                allowFullScreen
                title="Movie Trailer"
              />
            </div>
          )}
        </div>
      </div>
    </div>
  );
};

export default TrailerModal;
'''

with open("netflix-clone/frontend/src/components/TrailerModal.jsx", "w") as f:
    f.write(modal_content)

# frontend/src/App.jsx
app_content = '''import React, { useState } from 'react';
import Navbar from './components/Navbar';
import Hero from './components/Hero';
import Row from './components/Row';
import TrailerModal from './components/TrailerModal';
import useFetch from './hooks/useFetch';
import './index.css';

function App() {
  const [selectedMovieId, setSelectedMovieId] = useState(null);
  const [isModalOpen, setIsModalOpen] = useState(false);

  const { data: rows, loading, error } = useFetch('/api/content/rows');

  const handlePlayTrailer = (movieId) => {
    setSelectedMovieId(movieId);
    setIsModalOpen(true);
  };

  const handleCloseModal = () => {
    setIsModalOpen(false);
    setSelectedMovieId(null);
  };

  return (
    <div className="App bg-netflix-black min-h-screen">
      <Navbar />

      <Hero onPlayTrailer={handlePlayTrailer} />

      <div className="space-y-12 py-12">
        {loading && (
          <div className="text-white text-center text-xl">Loading content...</div>
        )}

        {error && (
          <div className="text-white text-center text-xl">Error loading content</div>
        )}

        {rows && rows.map((row, index) => (
          <Row
            key={index}
            title={row.title}
            movies={row.movies}
          />
        ))}
      </div>

      <TrailerModal
        movieId={selectedMovieId}
        isOpen={isModalOpen}
        onClose={handleCloseModal}
      />
    </div>
  );
}

export default App;
'''

with open("netflix-clone/frontend/src/App.jsx", "w") as f:
    f.write(app_content)

print("✅ React components created!")

⚛️ Creating React components...
✅ React components created!


In [86]:
# Step 7: Set environment variables
print("🔑 Setting environment variables...")

import os

# Set your TMDB API key (get free from https://www.themoviedb.org/settings/api)
os.environ["TMDB_API_KEYS"] = "88fe0e109ad1085b8130fe484b27b40a"  # REPLACE WITH YOUR ACTUAL KEY

print("✅ Environment variables set!")

🔑 Setting environment variables...
✅ Environment variables set!


In [87]:
# Step 8: Start backend server
print("🚀 Starting backend server...")

import subprocess
import time
import requests
import os

def start_backend():
    # Make sure we're in the right directory
    backend_path = "/content/netflix-clone/backend"
    if not os.path.exists(backend_path):
        print(f"❌ Backend directory not found: {backend_path}")
        return None

    os.chdir(backend_path)

    # Start the backend server
    try:
        process = subprocess.Popen([
            "python", "-m", "uvicorn", "app.main:app",
            "--host", "0.0.0.0", "--port", "8000"
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return process
    except Exception as e:
        print(f"❌ Failed to start backend: {e}")
        return None

# Start the backend
backend_process = start_backend()

if backend_process is None:
    print("❌ Backend failed to start. Checking for issues...")
    # Check if files exist
    print("📁 Checking backend files...")
    !find /content/netflix-clone/backend -name "*.py" | head -10
else:
    # Wait for backend to start
    print("⏳ Waiting for backend server to start...")
    time.sleep(10)

    # Check if backend is running
    def check_backend_ready():
        try:
            response = requests.get("http://localhost:8000/health", timeout=5)
            return response.status_code == 200
        except:
            return False

    if check_backend_ready():
        print("✅ Backend is running successfully!")

        # Create backend tunnel
        try:
            backend_tunnel = ngrok.connect(8000, "http")
            BACKEND_URL = backend_tunnel.public_url
            print(f"🌐 Backend Public URL: {BACKEND_URL}")
        except Exception as e:
            print(f"❌ Ngrok tunnel failed: {e}")
            BACKEND_URL = "http://localhost:8000"
            print(f"🔗 Using local backend: {BACKEND_URL}")
    else:
        print("❌ Backend is not responding. Checking logs...")
        # Try to get error output
        try:
            stdout, stderr = backend_process.communicate(timeout=2)
            if stderr:
                print(f"Backend error: {stderr.decode()}")
        except:
            print("Could not get backend logs")

        BACKEND_URL = "http://localhost:8000"
        print(f"⚠️  Using fallback URL: {BACKEND_URL}")

🚀 Starting backend server...
❌ Backend directory not found: /content/netflix-clone/backend
❌ Backend failed to start. Checking for issues...
📁 Checking backend files...
find: ‘/content/netflix-clone/backend’: No such file or directory


In [88]:
# Step 9: Setup frontend
print("🚀 Setting up frontend...")

import subprocess
import os

# Change to frontend directory
frontend_path = "/content/netflix-clone/frontend"
if not os.path.exists(frontend_path):
    print(f"❌ Frontend directory not found: {frontend_path}")
else:
    os.chdir(frontend_path)
    print(f"✅ Changed to frontend directory: {os.getcwd()}")

    # Check if package.json exists
    if not os.path.exists("package.json"):
        print("❌ package.json not found!")
        print("📁 Files in frontend directory:")
        !ls -la
    else:
        print("✅ package.json found")

    # Install dependencies
    print("📦 Installing dependencies...")
    try:
        # Run npm install with full path
        result = subprocess.run(
            ["npm", "install"],
            capture_output=True,
            text=True,
            timeout=120  # 2 minute timeout
        )

        if result.returncode == 0:
            print("✅ Dependencies installed successfully")
        else:
            print(f"❌ npm install failed with return code: {result.returncode}")
            if result.stderr:
                print(f"Error output: {result.stderr[:500]}")  # Show first 500 chars
    except subprocess.TimeoutExpired:
        print("❌ npm install timed out")
    except Exception as e:
        print(f"❌ npm install error: {e}")

    # Build the frontend
    print("🔨 Building frontend...")
    try:
        result = subprocess.run(
            ["npm", "run", "build"],
            capture_output=True,
            text=True,
            timeout=180  # 3 minute timeout
        )

        if result.returncode == 0:
            print("✅ Frontend built successfully")
            # Verify build was created
            if os.path.exists("dist"):
                print("✅ Build directory created")
                !ls -la dist/
            else:
                print("❌ Build directory 'dist' not found")
        else:
            print(f"❌ Build failed with return code: {result.returncode}")
            if result.stderr:
                print(f"Build error: {result.stderr[:500]}")
            if result.stdout:
                print(f"Build output: {result.stdout[:500]}")

    except subprocess.TimeoutExpired:
        print("❌ Build timed out")
    except Exception as e:
        print(f"❌ Build error: {e}")

🚀 Setting up frontend...
❌ Frontend directory not found: /content/netflix-clone/frontend


In [89]:
# Step 10: Start frontend server
print("🌐 Starting frontend server...")

import subprocess
import time
import os

def start_frontend():
    frontend_path = "/content/netflix-clone/frontend"

    # Check if frontend directory exists
    if not os.path.exists(frontend_path):
        print(f"❌ Frontend directory not found: {frontend_path}")
        return None

    # Check if build was created
    if not os.path.exists(f"{frontend_path}/dist"):
        print("❌ No build directory found. Please run Step 9 first.")
        return None

    os.chdir(frontend_path)

    # Start the frontend server
    try:
        print("Starting Vite preview server...")
        process = subprocess.Popen([
            "npx", "vite", "preview", "--host", "0.0.0.0", "--port", "3000"
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return process
    except Exception as e:
        print(f"❌ Failed to start frontend: {e}")
        return None

# Start the frontend server
frontend_process = start_frontend()

if frontend_process is None:
    print("❌ Frontend server failed to start")
    FRONTEND_URL = "http://localhost:3000"
else:
    # Wait for frontend to start
    print("⏳ Waiting for frontend server to start...")
    time.sleep(8)

    # Check if frontend is running
    def check_frontend_ready():
        try:
            import requests
            response = requests.get("http://localhost:3000", timeout=5)
            return response.status_code == 200
        except:
            return False

    if check_frontend_ready():
        print("✅ Frontend server is running!")

        # Create frontend tunnel
        try:
            frontend_tunnel = ngrok.connect(3000, "http")
            FRONTEND_URL = frontend_tunnel.public_url
            print(f"🌐 Frontend Public URL: {FRONTEND_URL}")
        except Exception as e:
            print(f"❌ Frontend tunnel failed: {e}")
            FRONTEND_URL = "http://localhost:3000"
            print(f"🔗 Using local frontend: {FRONTEND_URL}")
    else:
        print("❌ Frontend server is not responding")
        FRONTEND_URL = "http://localhost:3000"
        print(f"⚠️  Using fallback URL: {FRONTEND_URL}")

print(f"🔗 Frontend will be available at: {FRONTEND_URL}")

🌐 Starting frontend server...
❌ Frontend directory not found: /content/netflix-clone/frontend
❌ Frontend server failed to start
🔗 Frontend will be available at: http://localhost:3000


In [90]:
# Step 11: Test application
print("🧪 Testing application...")

# Test backend health
try:
    backend_response = requests.get(f"{BACKEND_URL}/health", timeout=10)
    if backend_response.status_code == 200:
        print(f"✅ Backend health: {backend_response.json()}")
    else:
        print(f"❌ Backend health check failed: {backend_response.status_code}")
except Exception as e:
    print(f"❌ Backend test failed: {e}")

# Test featured content
try:
    featured_response = requests.get(f"{BACKEND_URL}/api/content/featured", timeout=10)
    if featured_response.status_code == 200:
        featured_data = featured_response.json()
        print(f"✅ Featured movie: {featured_data.get('title', 'Loaded successfully')}")
    else:
        print(f"❌ Featured content failed: {featured_response.status_code}")
except Exception as e:
    print(f"❌ Featured content test failed: {e}")

# Test rows content
try:
    rows_response = requests.get(f"{BACKEND_URL}/api/content/rows", timeout=10)
    if rows_response.status_code == 200:
        rows_data = rows_response.json()
        print(f"✅ Loaded {len(rows_data)} content rows")
    else:
        print(f"❌ Rows content failed: {rows_response.status_code}")
except Exception as e:
    print(f"❌ Rows content test failed: {e}")

🧪 Testing application...
✅ Backend health: {'status': 'healthy'}
❌ Featured content failed: 404
✅ Loaded 0 content rows


In [91]:
# Step 12: Display final results
print("\n" + "="*60)
print("🎉 NETFLIX CLONE SUCCESSFULLY DEPLOYED!")
print("="*60)
print(f"📱 FRONTEND: {FRONTEND_URL}")
print(f"🔧 BACKEND:  {BACKEND_URL}")
print(f"📚 API DOCS: {BACKEND_URL}/docs")
print("="*60)
print("\n🎬 **NEXT STEPS:**")
print("1. 📱 Open the FRONTEND URL in your browser")
print("2. 🎥 Click 'Play' on any movie to test trailers")
print("3. 📜 Scroll through movie rows")
print("4. 🔍 Check the API docs for backend endpoints")
print("\n⚠️  **IMPORTANT:**")
print("   • Keep this Colab tab open to keep servers running")
print("   • Free Colab sessions last ~12 hours")
print("   • To restart: Run all cells from beginning")
print("="*60)


🎉 NETFLIX CLONE SUCCESSFULLY DEPLOYED!
📱 FRONTEND: http://localhost:3000
🔧 BACKEND:  http://localhost:8000
📚 API DOCS: http://localhost:8000/docs

🎬 **NEXT STEPS:**
1. 📱 Open the FRONTEND URL in your browser
2. 🎥 Click 'Play' on any movie to test trailers
3. 📜 Scroll through movie rows
4. 🔍 Check the API docs for backend endpoints

⚠️  **IMPORTANT:**
   • Keep this Colab tab open to keep servers running
   • Free Colab sessions last ~12 hours
   • To restart: Run all cells from beginning


In [92]:
# Step 13: Keep servers alive
print("🔋 Keeping servers alive...")

from IPython.display import display, Javascript
import threading

def keep_alive():
    """Prevent Colab from timing out"""
    import time
    while True:
        time.sleep(60)
        print("💚 Servers still running...", end='\r')

# Start keep-alive in background
keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()

# JavaScript to prevent browser timeout
display(Javascript("""
function KeepAlive() {
    console.log('Netflix Clone - Keeping session alive');
}
setInterval(KeepAlive, 60000);
"""))

print("✅ Servers are running and will stay alive")
print("🛑 To stop servers: Restart runtime or close tab")

🔋 Keeping servers alive...


<IPython.core.display.Javascript object>

✅ Servers are running and will stay alive
🛑 To stop servers: Restart runtime or close tab
